In [1]:
from huggingface_hub import list_repo_refs

refs = list_repo_refs("livekit/turn-detector")
print("Branches:", [b.name for b in refs.branches])
print("Tags:", [t.name for t in refs.tags])

Branches: ['main', 'add-4.1', 'distilled-model', 'normalized-model-v3', 'normalized-model', 'z123', 'x123', 'q3', 'dev-punc-dropout', 'multlingual', 'develop']
Tags: ['v0.4.1-intl', 'v0.4.0-intl', 'v0.3.1-intl', 'v0.3.0-intl', 'v0.2.1-intl', 'v0.2.0-intl', 'v1.2.2-en', 'v0.1.0-intl', 'v1.2.1', 'v1.2.0']


In [2]:
from huggingface_hub import snapshot_download
path = snapshot_download("livekit/turn-detector", revision="v0.4.1-intl", local_files_only=True)
print(path)

/Users/rakhmonberdiev/.cache/huggingface/hub/models--livekit--turn-detector/snapshots/87e35fcb1e60a569bea70346191c4886ea92e281


In [3]:
import json, time, unicodedata, re
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

HG_MODEL = "livekit/turn-detector"
ONNX_FILENAME = "model_q8.onnx"
MODEL_REVISION = "v0.4.1-intl"
MAX_HISTORY_TOKENS = 128
TEST_LANGUAGE = "en"

onnx_path = hf_hub_download(HG_MODEL, ONNX_FILENAME, subfolder="onnx",
                           revision=MODEL_REVISION, local_files_only=True)
lang_path = hf_hub_download(HG_MODEL, "languages.json",
                           revision=MODEL_REVISION, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(HG_MODEL, revision=MODEL_REVISION,
                                          local_files_only=True, truncation_side="left")
session = ort.InferenceSession(onnx_path, providers=["CoreMLExecutionProvider", "CPUExecutionProvider"])


tok = tokenizer
IM_END_ID = tok.convert_tokens_to_ids("<|im_end|>")
with open(lang_path) as f:
    languages = json.load(f)
threshold = languages.get(TEST_LANGUAGE, {}).get("threshold", 0.5)


def build_text(messages):
    convo = tok.apply_chat_template(messages, add_generation_prompt=False,
                                    add_special_tokens=False, tokenize=False)
    return convo[:convo.rfind("<|im_end|>")]

print(f"Ready. threshold={threshold:.4f}, im_end_id={IM_END_ID}")

Ready. threshold=0.0110, im_end_id=151645


2026-07-31 14:35:01.578 Python[1179:11216] 2026-07-31 14:35:01.578223 [W:onnxruntime:, coreml_execution_provider.cc:137 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 195 number of nodes in the graph: 3445 number of nodes supported by CoreML: 1695


In [4]:
print(f"Language: {TEST_LANGUAGE}, threshold={threshold:.4f}")
print(f"Tillar: {languages}") 

Language: en, threshold=0.0110
Tillar: {'zh': {'threshold': 0.0066, 'tpr': 0.9933, 'tnr': 0.8661}, 'de': {'threshold': 0.0062, 'tpr': 0.9931, 'tnr': 0.8777}, 'nl': {'threshold': 0.0077, 'tpr': 0.9931, 'tnr': 0.8807}, 'en': {'threshold': 0.011, 'tpr': 0.993, 'tnr': 0.8696}, 'pt': {'threshold': 0.0069, 'tpr': 0.9932, 'tnr': 0.8745}, 'es': {'threshold': 0.0058, 'tpr': 0.9931, 'tnr': 0.8603}, 'fr': {'threshold': 0.0078, 'tpr': 0.993, 'tnr': 0.8892}, 'it': {'threshold': 0.0037, 'tpr': 0.993, 'tnr': 0.851}, 'ja': {'threshold': 0.0096, 'tpr': 0.9934, 'tnr': 0.888}, 'ko': {'threshold': 0.0156, 'tpr': 0.9937, 'tnr': 0.9449}, 'ru': {'threshold': 0.0032, 'tpr': 0.9933, 'tnr': 0.8803}, 'tr': {'threshold': 0.0045, 'tpr': 0.9934, 'tnr': 0.8727}, 'id': {'threshold': 0.0132, 'tpr': 0.993, 'tnr': 0.8943}, 'hi': {'threshold': 0.0398, 'tpr': 0.994, 'tnr': 0.9627}}


In [5]:
import json, torch
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

MODEL, REV, MAX_LEN = "livekit/turn-detector", "v0.4.1-intl", 128

class EOUDataset(Dataset):
    def __init__(self, path):
        self.rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        enc = tok(build_text(r["messages"]), add_special_tokens=False,
                  truncation=True, max_length=MAX_LEN)
        return {"input_ids": enc["input_ids"], "label": float(r["label"])}

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    ids_out, attn, labels = [], [], []
    for b in batch:
        ids = b["input_ids"]; pad = maxlen - len(ids)
        ids_out.append([tok.pad_token_id] * pad + ids)   # LEFT padding — muhim
        attn.append([0] * pad + [1] * len(ids))
        labels.append(b["label"])
    return {"input_ids": torch.tensor(ids_out),
            "attention_mask": torch.tensor(attn),
            "labels": torch.tensor(labels)}

class EOUTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        logprobs = torch.log_softmax(out.logits[:, -1, :].float(), dim=-1)
        p = logprobs[:, IM_END_ID].exp().clamp(1e-6, 1 - 1e-6)
        loss = -(labels * p.log() + (1 - labels) * (1 - p).log()).mean()
        return (loss, out) if return_outputs else loss

In [6]:
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, dtype=torch.float32)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],
    task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 259,774,336 || trainable%: 0.4163


In [7]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./uz_eou_lora/v1_R",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=3e-5,
    warmup_ratio=0.05,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True, 
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    weight_decay=0.01,
    report_to="none",
)

trainer = EOUTrainer(
    model=model, args=args,
    train_dataset=EOUDataset("./uz_dataset/uz_eou_train_num.jsonl"),
    eval_dataset=EOUDataset("./uz_dataset/uz_eou_val_num.jsonl"),
    data_collator=collate,

)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/rakhmonberdiev/Research/TurnDetecting/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.557208,0.557252
2,0.402242,0.443943
3,0.351338,0.398430


[transformers] You are using a model of type `llama` to instantiate a model of type `qwen2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
/Users/rakhmonberdiev/Research/TurnDetecting/.venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/Users/rakhmonberdiev/Research/TurnDetecting/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] You are using a model of type `llama` to instantiate a model of type `qwen2`. This may be expect

TrainOutput(global_step=2190, training_loss=0.5070702071603574, metrics={'train_runtime': 2024.5264, 'train_samples_per_second': 17.306, 'train_steps_per_second': 1.082, 'total_flos': 3077756372256000.0, 'train_loss': 0.5070702071603574, 'epoch': 3.0})

In [8]:
import json
with open("./uz_eou_lora/v2/checkpoint-1584/trainer_state.json") as f:
    st = json.load(f)
print("epoch:", st["epoch"])
print("global_step:", st["global_step"], "/", st.get("max_steps"))
print("best:", st.get("best_metric"))
print("\noxirgi loglar:")
for e in st["log_history"][-6:]:
    print(e)

epoch: 4.0
global_step: 1584 / 1980
best: 0.09456887096166611

oxirgi loglar:
{'epoch': 3.724747474747475, 'grad_norm': 0.0002742684737313539, 'learning_rate': 1.6834246596745494e-05, 'loss': 1.1007294524461031e-05, 'step': 1475}
{'epoch': 3.787878787878788, 'grad_norm': 0.0004351535171736032, 'learning_rate': 1.530048699286488e-05, 'loss': 0.0004522816091775894, 'step': 1500}
{'epoch': 3.851010101010101, 'grad_norm': 0.010578688234090805, 'learning_rate': 1.3827278801600979e-05, 'loss': 4.427147563546896e-05, 'step': 1525}
{'epoch': 3.9141414141414144, 'grad_norm': 0.00010286088945576921, 'learning_rate': 1.2417192803616934e-05, 'loss': 1.635748310945928e-05, 'step': 1550}
{'epoch': 3.9772727272727275, 'grad_norm': 0.00022515005548484623, 'learning_rate': 1.1072689629961696e-05, 'loss': 1.6194379422813654e-05, 'step': 1575}
{'epoch': 4.0, 'eval_loss': 0.10723382234573364, 'eval_runtime': 39.6012, 'eval_samples_per_second': 17.752, 'eval_steps_per_second': 0.556, 'step': 1584}


In [9]:
import json
st = json.load(open("./uz_eou_lora/v6_numbers/checkpoint-2076/trainer_state.json"))
print("best checkpoint:", st.get("best_model_checkpoint"))
print("best metric:", st.get("best_metric"))
# barcha eval nuqtalari
for e in st["log_history"]:
    if "eval_loss" in e:
        print(f"  epoch {e['epoch']:.0f}: eval_loss {e['eval_loss']:.5f}")

best checkpoint: ./uz_eou_lora/v6_numbers/checkpoint-2076
best metric: 0.16060097515583038
  epoch 1: eval_loss 0.24158
  epoch 2: eval_loss 0.17584
  epoch 3: eval_loss 0.16060
